# PayRevive failure diagnosis

For a completed failure, a specific reported reason supplies the diagnosis. When the reason is missing or generic, the experiment compares three classifiers on the ambiguous subset. It scores a probability distribution over six synthetic causes, including unknown. An unknown prediction or insufficient score produces an abstention.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import Image, display

from ml.analysis.diagnosis import run_diagnosis

dataset = Path(os.environ.get("PAYREVIVE_NOTEBOOK_DATASET", "work/datasets/demo"))
output = Path(
    os.environ.get("PAYREVIVE_NOTEBOOK_OUTPUT", f"work/analysis/interactive-{uuid4().hex[:8]}")
)
challenge = os.environ.get("PAYREVIVE_NOTEBOOK_CHALLENGE")
report = run_diagnosis(
    dataset,
    output,
    os.environ.get("PAYREVIVE_EXPECTED_MANIFEST"),
    Path(challenge) if challenge else None,
)
display(pd.DataFrame.from_dict(report["cohorts"], orient="index"))
print(report["months"])
print("Dataset manifest", report["provenance"]["source_manifest_sha256"])

## Temporal comparison

The first month fits preprocessing and models. The second month chooses between logistic regression and a random forest using macro F1. The most frequent class provides a baseline. The third month measures performance after those choices are fixed. Macro F1 weights each class equally, accuracy weights each example equally, and log loss measures the quality of the full probability vector.

Customer history is excluded from this version because the first month has no earlier outcomes. Latency is available after the response. Expiry, routing, billing context, and the other features come from the shared allowlist. The simulator supplies ground-truth labels to evaluate ambiguous failures.

In [ ]:
results = {"validation": report["validation"], "test": report["test"]}
if "challenge" in report:
    results["challenge"] = report["challenge"]["comparison"]
display(
    pd.DataFrame(
        [
            {
                "split": split,
                "model": name,
                **{k: row[k] for k in ("accuracy", "macro_f1", "log_loss")},
            }
            for split, metrics in results.items()
            for name, row in metrics.items()
        ]
    )
)
selected = report["selected_model"]
test = report["test"][selected]
display(pd.DataFrame({"f1": test["per_class_f1"], "support": test["class_support"]}))
display(pd.DataFrame(test["confusion_matrix"], index=report["classes"], columns=report["classes"]))

## Abstention

The validation sweep targets at least 70% accuracy among accepted predictions and at least 10% coverage of ambiguous failures. If no threshold meets both criteria, the simulator abstains on ambiguous cases. The validation criterion is a heuristic operating point. Test outcomes can fall below it.

In [ ]:
display(pd.DataFrame(report["validation_thresholds"]))
print("Selected model", selected)
print("Selected threshold", report["selected_threshold"])
display(pd.Series(report["test_selective"], name="test"))
if "challenge" in report:
    display(pd.Series(report["challenge"]["selective"], name="challenge"))

## SHAP in probability space

Permutation SHAP explains the selected pipeline using a sample of training failures as the background distribution. A positive contribution increases the predicted probability of the displayed class relative to the background. The sum of all contributions plus the background value reconstructs the model output. The estimates depend on the model, masked feature combinations, and background sample. Use this chart to audit a prediction and its inputs. Causal conclusions require separate evidence.

In [ ]:
display(Image(filename=str(output / "shap.png")))
display(
    pd.Series(
        report["shap"]["mean_absolute_attribution"], name="mean absolute attribution"
    ).sort_values(ascending=False)
)
print("Maximum probability reconstruction error", report["shap"]["max_additivity_error"])